In [ ]:
from PIL import Image
import torch
import numpy as np
import glob

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
from monai.inferers import SlidingWindowInferer

from skimage.morphology import label
from utils import model_fn
from image_processing import (
    normalize,
    postprocessing_sdt,
    last_layer_fn,
)

In [ ]:
model_path = r"last_epoch_model.pth"
image_path_dir = r'data\test_images'

test_images_filename =[]
for file in glob.glob(image_path_dir + "\*.tif"):
    test_images_filename.append(os.path.basename(file))

print (test_images_filename)

In [ ]:
# Load image and normalize
def load_image (image_path_dir, filename):
    image = np.array(Image.open(os.path.join(image_path_dir, filename))).astype(np.float32)
    image = normalize(image)
    return image

def predict_image (image, device, inferer, model):
    # Transform
    data = torch.tensor(image).to(device)[(None,)*2] # unsqueeze(0) twice
    print(data.shape)
# Set model to eval
    model.eval()
    with torch.no_grad():
        pred = inferer(inputs=data, network=model)
    # tensor to numpy
    #output = pred[:, 4, :, :].squeeze().detach().cpu().numpy()
    pred_instance_fibre = postprocessing_sdt(pred[:, 3, :, :])
    pred_instance_axon = postprocessing_sdt(pred[:, 4, :, :])
    pred_semantic = last_layer_fn(pred[:, 0:3, :, :]).squeeze().detach().cpu().numpy()
    return pred_instance_fibre, pred_instance_axon, pred_semantic

In [ ]:
# Create the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model_fn(device)

# Load state_dict
model.load_state_dict(torch.load(model_path))

# inferer
inferer = SlidingWindowInferer(roi_size=(1024, 1024), sw_batch_size=1, overlap=0.25, progress=True)

In [ ]:
for filename in test_images_filename:
    image = load_image (image_path_dir, filename)
    pred_instance_fibre, pred_instance_axon, pred_semantic = predict_image (image, device, inferer, model)
    
    # get inne_myelin and axon instances
    inner_myelin_instance = label(pred_semantic >= 2)
    # save images
    image_pred_fibre = Image.fromarray(pred_instance_fibre.astype(np.uint16), 'I;16') # 16-bit
    image_pred_fibre.save(os.path.join(r'test_output\fibre', filename))
    image_pred_semantic = Image.fromarray(pred_semantic.astype(np.uint16), 'I;16') # 16-bit
    image_pred_semantic.save(os.path.join(r'test_output\semantic', filename))
    image_pred_inner_myelin = Image.fromarray(inner_myelin_instance.astype(np.uint16), 'I;16') # 16-bit
    image_pred_inner_myelin.save(os.path.join(r'test_output\inner_myelin', filename))
    image_pred_axon = Image.fromarray(pred_instance_axon.astype(np.uint16), 'I;16') # 16-bit
    image_pred_axon.save(os.path.join(r'test_output\axon', filename))